In [2]:
import random
import math

def sigmoid(z):
    z = max(-500, min(500, z))
    return 1.0 / (1.0 + math.exp(-z))


random.seed(42)
N = 200
X = []
y = []

for _ in range(N // 2):
    X.append([random.gauss(2, 1), random.gauss(2, 1)])
    y.append(0)

for _ in range(N // 2):
    X.append([random.gauss(5, 1), random.gauss(5, 1)])
    y.append(1)

combined = list(zip(X, y))
random.shuffle(combined)
X, y = zip(*combined)
X = list(X)
y = list(y)

print(f"Generated {N} samples (2 classes, 2 features)")
print(f"Class 0 center: (2, 2), Class 1 center: (5, 5)")
print(f"First 5 samples:")
for i in range(5):
    print(f"  Features: [{X[i][0]:.2f}, {X[i][1]:.2f}], Label: {y[i]}")

Generated 200 samples (2 classes, 2 features)
Class 0 center: (2, 2), Class 1 center: (5, 5)
First 5 samples:
  Features: [5.67, 3.86], Label: 1
  Features: [0.88, 2.46], Label: 0
  Features: [1.53, 2.50], Label: 0
  Features: [4.60, 6.20], Label: 1
  Features: [6.30, 6.90], Label: 1


Logistic Regression in NumPy

In [3]:
import numpy as np

class LogisticRegression:
    def __init__(self, n_features, learning_rate=0.01):
        self.weights = np.zeros(n_features)
        self.bias = 0.0
        self.lr = learning_rate
        self.loss_history = []

    def _sigmoid(self, z):
        return 1 / (1 + np.exp(-z))

    def predict_proba(self, X):
        z = X @ self.weights + self.bias
        return self._sigmoid(z)

    def predict(self, X, threshold=0.5):
        return (self.predict_proba(X) >= threshold).astype(int)

    def compute_loss(self, X, y):
        n = len(y)
        p = self.predict_proba(X)
        p = np.clip(p, 1e-15, 1 - 1e-15)
        return -np.mean(y * np.log(p) + (1 - y) * np.log(1 - p))

    def fit(self, X, y, epochs=1000, print_every=200):
        X = np.asarray(X)
        y = np.asarray(y)
        n = len(y)

        for epoch in range(epochs):
            p = self.predict_proba(X)
            error = p - y

            dw = (X.T @ error) / n
            db = np.mean(error)

            self.weights -= self.lr * dw
            self.bias    -= self.lr * db

            loss = self.compute_loss(X, y)
            self.loss_history.append(loss)

            if epoch % print_every == 0:
                print(f"  Epoch {epoch:4d} | Loss: {loss:.4f} | "
                      f"w: [{self.weights[0]:.3f}, {self.weights[1]:.3f}] | "
                      f"b: {self.bias:.3f}")
        return self

    def accuracy(self, X, y):
        preds = self.predict(X)
        return np.mean(preds == y)


# ---- Train / test split ----
split = int(0.8 * N)
X_train, X_test = X[:split], X[split:]
y_train, y_test = y[:split], y[split:]

print("\n=== Training Logistic Regression ===")
model = LogisticRegression(n_features=2, learning_rate=0.1)
model.fit(X_train, y_train, epochs=1000, print_every=200)

print(f"\nTrain accuracy: {model.accuracy(X_train, y_train):.4f}")
print(f"Test accuracy:  {model.accuracy(X_test, y_test):.4f}")
print(f"Weights: [{model.weights[0]:.4f}, {model.weights[1]:.4f}]")
print(f"Bias: {model.bias:.4f}")


=== Training Logistic Regression ===
  Epoch    0 | Loss: 0.6289 | w: [0.074, 0.070] | b: -0.001
  Epoch  200 | Loss: 0.3136 | w: [0.464, 0.333] | b: -2.402
  Epoch  400 | Loss: 0.2173 | w: [0.657, 0.497] | b: -3.756
  Epoch  600 | Loss: 0.1723 | w: [0.795, 0.608] | b: -4.684
  Epoch  800 | Loss: 0.1462 | w: [0.902, 0.693] | b: -5.391

Train accuracy: 0.9875
Test accuracy:  1.0000
Weights: [0.9886, 0.7629]
Bias: -5.9623


 Confusion matrix and metrics from scratch

In [4]:
import numpy as np

class ClassificationMetrics:
    def __init__(self, y_true, y_pred):
        y_true = np.asarray(y_true)
        y_pred = np.asarray(y_pred)

        self.tp = np.sum((y_true == 1) & (y_pred == 1))
        self.tn = np.sum((y_true == 0) & (y_pred == 0))
        self.fp = np.sum((y_true == 0) & (y_pred == 1))
        self.fn = np.sum((y_true == 1) & (y_pred == 0))

    def accuracy(self):
        total = self.tp + self.tn + self.fp + self.fn
        return (self.tp + self.tn) / total if total > 0 else 0.0

    def precision(self):
        denom = self.tp + self.fp
        return self.tp / denom if denom > 0 else 0.0

    def recall(self):
        denom = self.tp + self.fn
        return self.tp / denom if denom > 0 else 0.0

    def f1(self):
        p, r = self.precision(), self.recall()
        return 2 * p * r / (p + r) if (p + r) > 0 else 0.0

    def print_confusion_matrix(self):
        print(f"\n  Confusion Matrix:")
        print(f"                  Predicted")
        print(f"                  Pos   Neg")
        print(f"  Actual Pos     {self.tp:4d}  {self.fn:4d}")
        print(f"  Actual Neg     {self.fp:4d}  {self.tn:4d}")

    def print_report(self):
        self.print_confusion_matrix()
        print(f"\n  Accuracy:  {self.accuracy():.4f}")
        print(f"  Precision: {self.precision():.4f}")
        print(f"  Recall:    {self.recall():.4f}")
        print(f"  F1 Score:  {self.f1():.4f}")


# ---- Usage ----
y_pred_test = model.predict(X_test)          # vectorized, returns np array
print("\n=== Classification Report (Test Set) ===")
metrics = ClassificationMetrics(y_test, y_pred_test)
metrics.print_report()


=== Classification Report (Test Set) ===

  Confusion Matrix:
                  Predicted
                  Pos   Neg
  Actual Pos       21     0
  Actual Neg        0    19

  Accuracy:  1.0000
  Precision: 1.0000
  Recall:    1.0000
  F1 Score:  1.0000
